# 15장 통합과 발표 준비

이 노트북은 웹 교재 [15장 통합과 발표 준비](https://pidocs.aifusionlab.org/ai-python/ch15/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/class_rssi_20261104.csv
!ls *.csv

## 15.1 시스템 통합 점검표

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch15/15-1/)

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("fastapi") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "httpx"])

import re
from statistics import median

import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class Measure(BaseModel):
    node_id: str = Field(min_length=1, max_length=16)
    zone: str = Field(pattern=r"^D[1-4]$")
    rssi: int = Field(ge=-100, le=0)

app = FastAPI()
records = []

@app.post("/measure")
def post_measure(m: Measure):
    records.append(m.model_dump())
    return {"ok": True}

@app.get("/summary")
def get_summary():
    zones = sorted({r["zone"] for r in records})
    return {z: median(r["rssi"] for r in records if r["zone"] == z) for z in zones}

api = TestClient(app)
rssi = pd.read_csv("class_rssi_20261104.csv").groupby("zone").head(5)
results = {}

codes = [api.post("/measure", json={"node_id": r.node, "zone": r.zone, "rssi": int(r.rssi)}).status_code
         for r in rssi.itertuples()]
results["1. 전송 20건 모두 200"] = codes.count(200) == 20
results["2. 잘못된 값 422"] = api.post("/measure", json={"node_id": "X", "zone": "D9", "rssi": 5}).status_code == 422
server = api.get("/summary").json()
local = rssi.groupby("zone")["rssi"].median().to_dict()
results["3. 서버 요약 = pandas 요약"] = all(abs(server[z] - local[z]) < 1e-9 for z in local)
report = "; ".join(f"{z} 중앙값 {v:g} dBm" for z, v in server.items())      # AI 대신 요약 문장
nums = [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", report)]
allowed = set(server.values()) | {1.0, 2.0, 3.0, 4.0}
results["4. 보고 수치 검증"] = all(x in allowed for x in nums)

print("보고:", report)
for k, v in results.items():
    print("PASS" if v else "FAIL", k)

## 15.2 예외 처리

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch15/15-2/)

In [ ]:
def to_rssi(text):
    """문자열을 신호 세기 정수로 변환한다. 변환할 수 없거나 범위 밖이면 None"""
    try:
        v = int(text)
    except ValueError:
        return None
    if not -100 <= v <= 0:
        return None
    return v

for t in ["-55", "abc", "-55.5", "20", ""]:
    print(repr(t), "->", to_rssi(t))

In [ ]:
import time

calls = {"n": 0}

def flaky_send(body):
    """시험용 전송 함수: 처음 두 번은 연결 실패, 세 번째부터 성공"""
    calls["n"] += 1
    if calls["n"] <= 2:
        raise OSError("connection failed")
    return True

def send_with_retry(send, body, max_try=3, wait=0.2):
    for i in range(1, max_try + 1):
        try:
            send(body)
            print(f"{i}회차 성공")
            return True
        except OSError as e:
            print(f"{i}회차 실패: {e}")
            time.sleep(wait * i)            # 시도할수록 대기 시간 증가
    print("재시도 한도 초과, 기록을 보관하고 다음 측정으로 진행")
    return False

print(send_with_retry(flaky_send, "{}"))
calls["n"] = 0
print(send_with_retry(flaky_send, "{}", max_try=2))